# Network of Thrones: hybrid RAG + knowledge graph assistant

This notebook combines two complementary knowledge sources:

- **Unstructured lore** (`lore.txt`): chunking → Sentence Transformer embeddings → FAISS dense index, combined with BM25 lexical retrieval.
- **Structured facts** (`graph.trig`): RDF/SPARQL lookup through a dedicated tool.
- **Grounded generation**: a local Ollama model chooses the appropriate tool and answers only from retrieved evidence.

The retrieval stack is local. Model downloads require internet access only on the first run.


In [1]:
# Run once in a fresh Colab/Jupyter environment.
%pip install -q langchain langchain-community langchain-ollama langgraph rdflib \
    sentence-transformers faiss-cpu rank_bm25 numpy pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


## 1. Start the local language model

The installation commands below target Google Colab/Linux. On Windows or macOS, install Ollama from https://ollama.com and start it before continuing.


In [2]:
%system apt-get update -qq && apt-get install -y -qq zstd

["W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)",
 'Selecting previously unselected package zstd.',
 '(Reading database ... ',
 '(Reading database ... 5%',
 '(Reading database ... 10%',
 '(Reading database ... 15%',
 '(Reading database ... 20%',
 '(Reading database ... 25%',
 '(Reading database ... 30%',
 '(Reading database ... 35%',
 '(Reading database ... 40%',
 '(Reading database ... 45%',
 '(Reading database ... 50%',
 '(Reading database ... 55%',
 '(Reading database ... 60%',
 '(Reading database ... 65%',
 '(Reading database ... 70%',
 '(Reading database ... 75%',
 '(Reading database ... 80%',
 '(Reading database ... 85%',
 '(Reading database ... 90%',
 '(Reading database ... 95%',
 '(Reading database ... 100%',
 '(Reading database ... 122403 files and directories currently installed.)',
 'Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd

In [4]:
%system curl -fsSL https://ollama.com/install.sh | sh

['>>> Cleaning up old version at /usr/local/lib/ollama',
 '>>> Installing ollama to /usr/local',
 '>>> Downloading ollama-linux-amd64.tar.zst',
 '#=#=#                                                                         ',
 '##O#-#                                                                        ',
 '##O=#  #                                                                      ',
 '#=#=-#  #                                                                     ',
 '-#O#- #   #                                                                   ',
 '',
 '                                                                           0.0%',
 '                                                                           0.2%',
 '                                                                           0.4%',
 '                                                                           0.8%',
 '#                                                                          1.5%',
 '##            

In [5]:
import platform
import shutil
import subprocess
import time
import requests

OLLAMA_MODEL = "mistral-nemo"

if shutil.which("ollama") is None:
    if platform.system() == "Linux":
        subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
    else:
        raise RuntimeError("Install Ollama from https://ollama.com, then restart this notebook.")

def ensure_ollama_running():
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2).raise_for_status()
        return
    except requests.RequestException:
        subprocess.Popen(
            ["ollama", "serve"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
        time.sleep(5)

ensure_ollama_running()
subprocess.run(["ollama", "pull", OLLAMA_MODEL], check=True)
print(f"Ollama is ready with {OLLAMA_MODEL}.")


Ollama is ready with mistral-nemo.


## 2. Load and chunk the lore corpus

Overlapping chunks reduce the chance that a fact is split across chunk boundaries. Each chunk keeps metadata so retrieved evidence can be inspected and cited.


In [7]:
from pathlib import Path
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

LORE_PATH = Path("lore.txt")
GRAPH_PATH = Path("graph.trig")
CHUNK_SIZE = 500
CHUNK_OVERLAP = 75
TOP_K = 4

if not LORE_PATH.exists():
    raise FileNotFoundError("Upload lore.txt to the notebook working directory before continuing.")

source_docs = TextLoader(str(LORE_PATH), encoding="utf-8").load()
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_documents(source_docs)

for chunk_id, chunk in enumerate(chunks):
    chunk.metadata.update({"chunk_id": chunk_id, "source": LORE_PATH.name})

print({"source_documents": len(source_docs), "chunks": len(chunks)})


{'source_documents': 1, 'chunks': 739}


## 3. Embed and index the chunks

`all-MiniLM-L6-v2` maps both chunks and questions to normalized vectors. FAISS inner-product search then acts as cosine-similarity search. BM25 remains useful for exact names and rare terms, so the final retriever fuses both rankings with Reciprocal Rank Fusion (RRF).


In [8]:
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
from langchain_community.retrievers import BM25Retriever

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
encoder = SentenceTransformer(EMBEDDING_MODEL)
chunk_texts = [chunk.page_content for chunk in chunks]

chunk_embeddings = encoder.encode(
    chunk_texts,
    normalize_embeddings=True,
    show_progress_bar=True,
)
chunk_embeddings = np.asarray(chunk_embeddings, dtype="float32")

dense_index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
dense_index.add(chunk_embeddings)

bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = min(12, len(chunks))

print({
    "embedding_model": EMBEDDING_MODEL,
    "embedding_dimension": chunk_embeddings.shape[1],
    "indexed_chunks": dense_index.ntotal,
})


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

{'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2', 'embedding_dimension': 384, 'indexed_chunks': 739}


In [14]:
def dense_retrieve(query: str, candidate_k: int = 12):
    candidate_k = min(candidate_k, len(chunks))
    query_vector = encoder.encode(
        [query], normalize_embeddings=True, show_progress_bar=False
    ).astype("float32")
    scores, positions = dense_index.search(query_vector, candidate_k)
    return [
        {"chunk_id": int(position), "dense_score": float(score)}
        for score, position in zip(scores[0], positions[0])
        if position >= 0
    ]


def hybrid_retrieve(query: str, top_k: int = TOP_K, candidate_k: int = 12):
    '''Fuse semantic and lexical rankings with Reciprocal Rank Fusion.'''
    candidate_k = min(candidate_k, len(chunks))
    dense_hits = dense_retrieve(query, candidate_k)
    bm25_retriever.k = candidate_k
    lexical_docs = bm25_retriever.invoke(query)

    lexical_ids = [int(doc.metadata["chunk_id"]) for doc in lexical_docs]
    dense_scores = {hit["chunk_id"]: hit["dense_score"] for hit in dense_hits}
    fused = {}
    rrf_constant = 60

    for rank, hit in enumerate(dense_hits, start=1):
        fused[hit["chunk_id"]] = fused.get(hit["chunk_id"], 0.0) + 1 / (rrf_constant + rank)
    for rank, chunk_id in enumerate(lexical_ids, start=1):
        fused[chunk_id] = fused.get(chunk_id, 0.0) + 1 / (rrf_constant + rank)

    ranked_ids = sorted(fused, key=fused.get, reverse=True)[:top_k]
    return [
        {
            "rank": rank,
            "chunk_id": chunk_id,
            "rrf_score": fused[chunk_id],
            "dense_score": dense_scores.get(chunk_id),
            "source": chunks[chunk_id].metadata.get("source", LORE_PATH.name),
            "text": chunks[chunk_id].page_content,
        }
        for rank, chunk_id in enumerate(ranked_ids, start=1)
    ]


def retrieval_table(query: str, top_k: int = TOP_K):
    '''Human-readable retrieval diagnostics.'''
    return pd.DataFrame(hybrid_retrieve(query, top_k=top_k))[
        ["rank", "chunk_id", "rrf_score", "dense_score", "source", "text"]
    ]


retrieval_table("Who killed a member of House Stark?", top_k=3)


,rank,chunk_id,rrf_score,dense_score,source,text
0,1,666,0.030536,0.479221,lore.txt,"Robb Stark, commonly known as The Young Wolf, ..."
1,2,356,0.016393,0.535155,lore.txt,Rickard Stark is a male character.\nHe is from...
2,3,224,0.016393,NaN,lore.txt,Jasper Waynwood is a male character.\nHe appea...


## 4. Expose hybrid retrieval and the RDF graph as tools

The lore tool returns evidence labels (`[lore:chunk_id]`) so the model can cite its sources. The graph tool is reserved for explicit relationships and other structured facts.


In [15]:
import rdflib
from langchain_core.tools import tool
import re
import unicodedata


@tool
def search_lore(query: str) -> str:
    """Search biographies, narrative events, aliases, titles, culture and appearances."""
    hits = hybrid_retrieve(query, top_k=TOP_K)
    if not hits:
        return "No relevant lore was found."
    return "\n\n".join(
        f"[lore:{hit['chunk_id']}] {hit['text']}" for hit in hits
    )


rdf_dataset = None
if GRAPH_PATH.exists():
    rdf_dataset = rdflib.Dataset(default_union=True)
    rdf_dataset.parse(str(GRAPH_PATH), format="trig")

def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("’", "'").replace("‘", "'")
    return " ".join(text.casefold().split())

character_index = {}

if rdf_dataset is not None:
    for subject in rdf_dataset.subjects():
        local_name = str(subject).rsplit("#", 1)[-1]
        display_name = local_name.replace("_", " ")
        character_index[normalize_text(display_name)] = display_name


def character_from_question(question: str) -> str | None:
    normalized_question = normalize_text(question)

    candidates = sorted(
        character_index.items(),
        key=lambda item: len(item[0]),
        reverse=True,
    )

    for normalized_name, display_name in candidates:
        if re.search(
            rf"\b{re.escape(normalized_name)}\b",
            normalized_question,
        ):
            return display_name

    return None


@tool
def query_graph_db(character_name: str) -> str:
    """Return all explicit graph relationships for one character; pass only the character name."""
    if rdf_dataset is None:
        return "No structured data found."

    normalized = normalize_text(character_name).replace(" ", "_")
    rows = []

    for subject, predicate, obj in rdf_dataset.triples((None, None, None)):
        subject_name = normalize_text(
            str(subject).rsplit("#", 1)[-1]
        )

        if subject_name != normalized:
            continue

        relation = str(predicate).rsplit("#", 1)[-1]
        value = (
            str(obj)
            .rsplit("#", 1)[-1]
            .replace("_", " ")
        )

        rows.append(f"{relation}: {value}")

    rows = sorted(set(rows))

    if not rows:
        return f"No structured data found for {character_name}."

    return "\n".join(rows)


print("Hybrid lore and RDF graph tools are ready.")


Hybrid lore and RDF graph tools are ready.


## 5. Grounded local agent

The prompt separates retrieval policy from answer formatting. It also instructs the agent to cite evidence labels and to abstain when the available sources are insufficient.


In [19]:
import re
from langchain.agents import create_agent
from langchain_ollama import ChatOllama
from langchain_core.messages import ToolMessage
from pydantic import BaseModel, Field


class ListAnswer(BaseModel):
    items: list[str] = Field(
        default_factory=list,
        description=(
            "Only atomic values directly requested by the question and explicitly "
            "present in the evidence. No predicates, labels, sentences, explanations, "
            "Markdown, tool names or source references."
        ),
    )


class SentenceAnswer(BaseModel):
    sentence: str = Field(
        default="",
        description=(
            "One factual sentence of at most 30 words that directly answers the "
            "descriptive question using only supplied evidence."
        ),
    )


llm = ChatOllama(model=OLLAMA_MODEL, temperature=0)

retrieval_prompt = """
You are a retrieval agent for a Game of Thrones database.

For every factual Game of Thrones question, call search_lore, query_graph_db,
or both. Never answer from pretrained knowledge.

Retrieval policy:
- For descriptive questions such as "Who is X?", call both tools.
- Use search_lore for biographies, aliases, titles, culture and appearances.
- Use query_graph_db for kills, family, service, marriage and guardianship.
- For query_graph_db, pass only the character name, never the full question.
- Use both tools when both structured and unstructured evidence may help.
- Make no more than two calls to each tool.

Your prose answer is discarded. Retrieve the evidence required by the question.
"""

retrieval_agent = create_agent(
    model=llm,
    tools=[search_lore, query_graph_db],
    system_prompt=retrieval_prompt,
)

list_formatter = ChatOllama(
    model=OLLAMA_MODEL, temperature=0
).with_structured_output(ListAnswer, method="json_schema")

sentence_formatter = ChatOllama(
    model=OLLAMA_MODEL, temperature=0
).with_structured_output(SentenceAnswer, method="json_schema")

print("The assistant is ready.")


The assistant is ready.


In [20]:
import re


NO_DATA_MARKERS = (
    "no relevant lore was found",
    "no structured data found",
    "graph.trig is unavailable",
)


RELATION_PATTERNS = [
    (r"^who\s+killed\b", "killedBy"),
    (r"\b(killed by|killer|killers)\b", "killedBy"),
    (r"\b(kill|kills|killed|victim|victims)\b", "killed"),
    (r"\b(parent|parents|mother|father)\b", "childOf"),
    (r"\b(child|children|son|sons|daughter|daughters)\b", "parentOf"),
    (r"\b(sibling|siblings|brother|brothers|sister|sisters)\b", "siblingOf"),
    (r"\bserved by\b", "servedBy"),
    (r"\b(serves|serve|served)\b", "serves"),
    (r"\bguarded by\b", "guardedBy"),
    (r"\b(guardian of|guards)\b", "guardianOf"),
    (r"\b(married|marriage|engaged|spouse|husband|wife)\b",
     "marriedOrEngagedTo"),
]


def requested_relation(question: str) -> str | None:
    normalized = normalize_text(question)

    for pattern, predicate in RELATION_PATTERNS:
        if re.search(pattern, normalized):
            return predicate

    return None


def extract_relation_values(
    graph_evidence: str,
    predicate: str,
) -> list[str]:
    values = []

    for line in graph_evidence.splitlines():
        line = line.strip().lstrip("-").strip()

        if ":" not in line:
            continue

        relation, value = line.split(":", 1)

        if relation.strip() != predicate:
            continue

        value = value.strip()

        if value and value not in values:
            values.append(value)

    return values


def is_descriptive_question(question: str) -> bool:
    """Recognize profile questions such as 'Who is Jon Snow?'."""
    normalized = normalize_text(question)
    return bool(re.match(r"^(who|what)\s+(is|was)\b", normalized))


def ask_assistant(user_query: str) -> str:
    # First handle explicit RDF relationships deterministically.
    relation = requested_relation(user_query)
    character_name = character_from_question(user_query)

    if relation is not None and character_name is not None:
        graph_evidence = query_graph_db.invoke({
            "character_name": character_name
        })

        values = extract_relation_values(
            graph_evidence,
            relation,
        )

        return (
            ", ".join(values)
            if values
            else "No entity found in the database."
        )

    # For descriptive or non-explicit questions, let the agent select tools.
    retrieval_result = retrieval_agent.invoke(
        {
            "messages": [
                {"role": "user", "content": user_query}
            ]
        },
        config={"recursion_limit": 15},
    )

    evidence_parts = []

    for message in retrieval_result["messages"]:
        if not isinstance(message, ToolMessage):
            continue

        content = str(message.content).strip()

        if content and not any(
            marker in content.lower()
            for marker in NO_DATA_MARKERS
        ):
            evidence_parts.append(content)

    if not evidence_parts:
        return "No entity found in the database."

    evidence = "\n\n".join(evidence_parts)

    if is_descriptive_question(user_query):
        answer = sentence_formatter.invoke(
            f"""
Write one direct factual sentence of at most 30 words answering the question.
Use only the supplied evidence.
Do not mention tools, sources, labels or retrieval.
Do not add information absent from the evidence.

Question:
{user_query}

Evidence:
{evidence}
"""
        )

        sentence = answer.sentence.strip()

        return (
            sentence
            if sentence
            else "No entity found in the database."
        )

    answer = list_formatter.invoke(
        f"""
Return only the atomic values requested by the question.

Rules:
- Use only the supplied evidence.
- Return values only.
- Do not return predicates, explanations, sentences, Markdown, tool names
  or source labels.
- If no matching values exist, return an empty items list.

Question:
{user_query}

Evidence:
{evidence}
"""
    )

    items = list(dict.fromkeys(
        item.strip()
        for item in answer.items
        if item and item.strip()
    ))

    return (
        ", ".join(items)
        if items
        else "No entity found in the database."
    )

## 7. Ask questions

This loop streams agent events, shows which knowledge source is consulted, and prints the final grounded answer.


In [21]:
print("Welcome to the Network of Thrones Assistant. Type 'exit' or 'quit' to stop.")

while True:
    user_input = input("\nUser: ").strip()

    if not user_input:
        continue

    if user_input.lower() in {"exit", "quit"}:
        print("System shutting down. Valar Morghulis!")
        break

    try:
        print(f"\nAssistant: {ask_assistant(user_input)}")
    except Exception as error:
        print(f"Request failed: {error}")

Welcome to the Network of Thrones Assistant. Type 'exit' or 'quit' to stop.

User: who is jon snow?

Assistant: Jon Snow is a male character from Northmen culture, born in 283 AC.

User: Who are Jon Snow’s parents?

Assistant: Lyanna Stark, Rhaegar Targaryen

User: Which characters serve Daenerys Targaryen?

Assistant: No entity found in the database.

User: Who are Cersei Lannister’s children and siblings?

Assistant: Joffrey Baratheon, Myrcella Baratheon, Tommen Baratheon

User: Which characters serve Daenerys Targaryen?

Assistant: No entity found in the database.

User: Which characters were guarded by direwolves or dragons?

Assistant: Nymeria, Drogon, Rhaegal

User: Which characters were both killers and victims?

Assistant: Tywin Lannister, Gregor Clegane

User: exit
System shutting down. Valar Morghulis!


__________________

## 6. Retrieval evaluation

Evaluate retrieval separately from generation. Add representative questions and the IDs of chunks that you judge relevant after inspecting `retrieval_table`. Recall@k and reciprocal rank expose whether failures come from retrieval rather than generation.


In [ ]:
def evaluate_retrieval(test_cases, k=TOP_K):
    rows = []
    for case in test_cases:
        hits = hybrid_retrieve(case["query"], top_k=k)
        retrieved_ids = [hit["chunk_id"] for hit in hits]
        relevant = set(case["relevant_chunk_ids"])
        relevant_ranks = [
            rank for rank, chunk_id in enumerate(retrieved_ids, start=1)
            if chunk_id in relevant
        ]
        rows.append({
            "query": case["query"],
            "retrieved_chunk_ids": retrieved_ids,
            f"recall@{k}": len(relevant.intersection(retrieved_ids)) / len(relevant) if relevant else 0.0,
            "reciprocal_rank": 1 / min(relevant_ranks) if relevant_ranks else 0.0,
        })
    return pd.DataFrame(rows)


# Replace these examples with validated chunk IDs from your own lore.txt.
evaluation_cases = [
    {"query": "Who is Jon Snow?", "relevant_chunk_ids": [574]},
]

if evaluation_cases:
    display(evaluate_retrieval(evaluation_cases))
else:
    print("Add labelled evaluation_cases after inspecting retrieval_table(query).")
